In [ ]:
%matplotlib inline

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import orbax.checkpoint as ocp
import pandas as pd
import seaborn as sns
from flax import nnx
from scipy.linalg import sqrtm

import h5py

sys.path.insert(0, os.path.abspath("../"))
sys.path.insert(0, os.path.abspath("../../.."))
from optbnn.bnn.likelihoods import LikCE
from optbnn.bnn.nets.mlp import MLP
from optbnn.bnn.priors import OptimGaussianPrior
from optbnn.sgmcmc_bayes_net.pref_net import PrefNet
from optbnn.utils import util

from iqlpref.reward_models.q_mlp import load_QMLP

In [ ]:
def plot_samples(
    X,
    samples,
    var=None,
    color="xkcd:bluish",
    smooth_q=False,
    ax=None,
):
    if ax is None:
        ax = plt.gca()
    if samples.ndim > 2:
        samples = samples.squeeze()
    # n_keep = int(samples.shape[1] / 10) if n_keep is None else n_keep
    # keep_idx = np.random.permutation(samples.shape[1])[:n_keep]
    mu = samples.mean(1)
    if var is None:
        q = 97.5  ## corresponds to 2 stdevs in Gaussian
        # q = 99.99  ## corresponds to 3 std
        Q = np.percentile(samples, [100 - q, q], axis=1)
        ub, lb = Q[1, :], Q[0, :]
        # ub, lb = mu + 2 * samples.std(1), mu - 2 * samples.std(1)
        if smooth_q:
            lb = moving_average(lb)
            ub = moving_average(ub)
    else:
        ub = mu + 3 * np.sqrt(var)
        lb = mu - 3 * np.sqrt(var)
    ####
    ax.fill_between(X.flatten(), ub, lb, color=color, alpha=0.25, lw=0)
    # ax.plot(X, samples[:, keep_idx], color=color, alpha=0.8)
    # ax.plot(X, mu, color=color, label=label)

In [ ]:
# t0012_mlp_full_path = os.path.expanduser(
#     "~/busy-beeway/transformers/mr_rewards_bb/mr-BB_t0012-f8bedad9-0.0/best_model.ckpt"
# )
# t0012_mlp_reduced_paths = [
#     "~/busy-beeway/transformers/mr_rewards_bb/mr-BB_t0012-c1bcc6d4-0.999/best_model.ckpt",
#     "~/busy-beeway/transformers/mr_rewards_bb/mr-BB_t0012-75cf3e5f-0.999/best_model.ckpt",
#     "~/busy-beeway/transformers/mr_rewards_bb/mr-BB_t0012-320168ce-0.999/best_model.ckpt",
#     "~/busy-beeway/transformers/mr_rewards_bb/mr-BB_t0012-0f7bc566-0.999/best_model.ckpt",
#     "~/busy-beeway/transformers/mr_rewards_bb/mr-BB_t0012-2b5679ac-0.999/best_model.ckpt",
#     "~/busy-beeway/transformers/mr_rewards_bb/mr-BB_t0012-3638f262-0.999/best_model.ckpt",
#     "~/busy-beeway/transformers/mr_rewards_bb/mr-BB_t0012-1f84d191-0.999/best_model.ckpt",
#     "~/busy-beeway/transformers/mr_rewards_bb/mr-BB_t0012-6eb6349d-0.999/best_model.ckpt",
#     "~/busy-beeway/transformers/mr_rewards_bb/mr-BB_t0012-917b2c7e-0.999/best_model.ckpt",
#     "~/busy-beeway/transformers/mr_rewards_bb/mr-BB_t0012-2db8269d-0.999/best_model.ckpt",
# ]
# t0012_mlp_reduced_paths = [os.path.expanduser(i) for i in t0012_mlp_reduced_paths]

In [ ]:
# t0012_bnn_prior_path = "./../exp/reward_learning/bb_tuning_star/br-BB_t0012-256-3"
# t0012_bnn_ckpt_path = os.path.join(
#     t0012_bnn_prior_path, "ckpts", "it-{}.ckpt".format(300)
# )
# t0012_bnn_full_path = (
#     "./../exp/reward_learning/bb_optim/br-bb_t0012-a76832c6/sampling_optim"
# )

# t0012_bnn_full_map = os.path.join(
#     t0012_bnn_full_path, "sampled_weights", "map_estimate"
# )

# t0012_bnn_reduced_maps = [
#     "./../exp/reward_learning/bb_optim/br-BB_t0012-4db14724-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0012-5cb6ae5f-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0012-30b649b7-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0012-90faf0aa-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0012-7144d9c0-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0012-05755075-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0012-a3126fff-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0012-ce3cc90b-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0012-d5e9136d-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0012-f5ef2594-0.999/sampling_optim/sampled_weights/map_estimate",
# ]

# t0012_t0048_bnn_reduced_maps = [
#     "./../exp/reward_learning/bb_optim/br-BB_t0012_t0048-4d35c0b8-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0012_t0048-5d111ee7-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0012_t0048-723d9d8f-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0012_t0048-85045ac8-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0012_t0048-412061a7-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0012_t0048-15965080-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0012_t0048-a8c9d5f3-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0012_t0048-ba576158-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0012_t0048-db560d7f-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0012_t0048-fc2d5992-0.999/sampling_optim/sampled_weights/map_estimate",
# ]

In [ ]:
# t0048_mlp_full_path = os.path.expanduser(
#     "~/busy-beeway/transformers/mr_rewards_bb/mr-BB_t0048-9c43669c-0.0/best_model.ckpt"
# )
# t0048_mlp_reduced_paths = [
#     "~/busy-beeway/transformers/mr_rewards_bb/mr-BB_t0048-50a33dd6-0.999/best_model.ckpt",
#     "~/busy-beeway/transformers/mr_rewards_bb/mr-BB_t0048-248678fb-0.999/best_model.ckpt",
#     "~/busy-beeway/transformers/mr_rewards_bb/mr-BB_t0048-b6329d3e-0.999/best_model.ckpt",
#     "~/busy-beeway/transformers/mr_rewards_bb/mr-BB_t0048-f4c90000-0.999/best_model.ckpt",
#     "~/busy-beeway/transformers/mr_rewards_bb/mr-BB_t0048-42635306-0.999/best_model.ckpt",
#     "~/busy-beeway/transformers/mr_rewards_bb/mr-BB_t0048-7632bf7e-0.999/best_model.ckpt",
#     "~/busy-beeway/transformers/mr_rewards_bb/mr-BB_t0048-5f04aad8-0.999/best_model.ckpt",
#     "~/busy-beeway/transformers/mr_rewards_bb/mr-BB_t0048-f544b024-0.999/best_model.ckpt",
#     "~/busy-beeway/transformers/mr_rewards_bb/mr-BB_t0048-bc6382c8-0.999/best_model.ckpt",
#     "~/busy-beeway/transformers/mr_rewards_bb/mr-BB_t0048-72d7e3a0-0.999/best_model.ckpt",
# ]
# t0048_mlp_reduced_paths = [os.path.expanduser(i) for i in t0048_mlp_reduced_paths]
# checkpointer = ocp.Checkpointer(ocp.CompositeCheckpointHandler())

In [ ]:
# t0048_bnn_prior_path = "./../exp/reward_learning/bb_tuning_star/br-BB_t0048-256-3"
# t0048_bnn_ckpt_path = os.path.join(
#     t0048_bnn_prior_path, "ckpts", "it-{}.ckpt".format(300)
# )
# t0048_bnn_full_path = (
#     "./../exp/reward_learning/bb_optim/br-BB_t0048-384f3849/sampling_optim"
# )

# t0048_bnn_full_map = os.path.join(
#     t0048_bnn_full_path, "sampled_weights", "map_estimate"
# )

# t0048_bnn_reduced_maps = [
#     "./../exp/reward_learning/bb_optim/br-BB_t0048-2ba4be8e-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0048-2befaba9-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0048-5f4731b4-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0048-9dd3a9e3-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0048-79f528b7-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0048-9443035a-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0048-a4ee12f1-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0048-d4ede0d2-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0048-f10bdc8b-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0048-fbed8ce7-0.999/sampling_optim/sampled_weights/map_estimate",
# ]

# t0048_t0012_bnn_reduced_maps = [
#     "./../exp/reward_learning/bb_optim/br-BB_t0048_t0012-02fca7c7-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0048_t0012-4f297476-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0048_t0012-8fb4318a-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0048_t0012-57fb20ee-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0048_t0012-1534a2a6-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0048_t0012-29170300-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0048_t0012-af05ee83-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0048_t0012-c3283252-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0048_t0012-dd233b8d-0.999/sampling_optim/sampled_weights/map_estimate",
#     "./../exp/reward_learning/bb_optim/br-BB_t0048_t0012-ec96c397-0.999/sampling_optim/sampled_weights/map_estimate",
# ]

In [ ]:
# t0012_mlp_full = nnx.jit(load_QMLP(t0012_mlp_full_path, checkpointer, on_cpu=True))
# t0012_mlp_rs = [
#     nnx.jit(load_QMLP(i, checkpointer, on_cpu=True)) for i in t0012_mlp_reduced_paths
# ]

# width = 256
# depth = 3
# transfer_fn = "relu"
# t0012_prior = OptimGaussianPrior(t0012_bnn_ckpt_path)
# t0012_net = MLP(28, 1, [width] * depth, transfer_fn)
# t0012_likelihood = LikCE()
# t0012_bayes_net = PrefNet(
#     t0012_net,
#     t0012_likelihood,
#     t0012_prior,
#     t0012_bnn_full_path,
#     n_gpu=0,
# )

# t0048_mlp_full = nnx.jit(load_QMLP(t0048_mlp_full_path, checkpointer, on_cpu=True))
# t0048_mlp_rs = [
#     nnx.jit(load_QMLP(i, checkpointer, on_cpu=True)) for i in t0048_mlp_reduced_paths
# ]

# t0048_prior = OptimGaussianPrior(t0048_bnn_ckpt_path)
# t0048_net = MLP(28, 1, [width] * depth, transfer_fn)
# t0048_likelihood = LikCE()
# t0048_bayes_net = PrefNet(
#     t0048_net,
#     t0048_likelihood,
#     t0048_prior,
#     t0048_bnn_full_path,
#     n_gpu=0,
# )
# with h5py.File("./../data/bb/state_data_1/t0029.hdf5") as f:
#     rng = np.random.default_rng(4)
#     n_samps = 1000
#     discount = 0.99
#     t0012_mlp_can_full = []
#     t0012_mlp_can_rs = []

#     t0012_bnn_can_full = []
#     t0012_bnn_can_rs = []

#     t0012_t0048_bnn_can_rs = []

#     t0048_mlp_can_full = []
#     t0048_mlp_can_rs = []

#     t0048_bnn_can_full = []
#     t0048_bnn_can_rs = []

#     t0048_t0012_bnn_can_rs = []

#     for ob in range(f["states"].shape[0]):
#         am = f["attn_mask"][ob]
#         if am.sum() > 2:
#             sts = f["states"][ob][: int(am.sum()), ...]
#             acts = f["actions"][ob][: int(am.sum()), ...]

#             t0012_mlp_preds_full = t0012_mlp_full(sts, acts).astype(np.float32)

#             t0012_mlp_preds_rs = [i(sts, acts).astype(np.float32) for i in t0012_mlp_rs]

#             t0012_bayes_net.load_map(t0012_bnn_full_map)
#             t0012_bnn_preds_full = (
#                 t0012_bayes_net.predict(
#                     np.concatenate([sts, acts], axis=-1), map_only=True
#                 )
#                 .astype(np.float32)
#                 .squeeze()
#             )

#             t0012_bnn_preds_rs = []
#             for i in t0012_bnn_reduced_maps:
#                 t0012_bayes_net.load_map(i)
#                 t0012_bnn_preds_rs.append(
#                     t0012_bayes_net.predict(
#                         np.concatenate([sts, acts], axis=-1), map_only=True
#                     )
#                     .astype(np.float32)
#                     .squeeze()
#                 )

#             t0012_t0048_bnn_preds_rs = []
#             for i in t0012_t0048_bnn_reduced_maps:
#                 t0012_bayes_net.load_map(i)
#                 t0012_t0048_bnn_preds_rs.append(
#                     t0012_bayes_net.predict(
#                         np.concatenate([sts, acts], axis=-1), map_only=True
#                     )
#                     .astype(np.float32)
#                     .squeeze()
#                 )

#             t0048_mlp_preds_full = t0048_mlp_full(sts, acts).astype(np.float32)

#             t0048_mlp_preds_rs = [i(sts, acts).astype(np.float32) for i in t0048_mlp_rs]

#             t0048_bayes_net.load_map(t0048_bnn_full_map)
#             t0048_bnn_preds_full = (
#                 t0048_bayes_net.predict(
#                     np.concatenate([sts, acts], axis=-1), map_only=True
#                 )
#                 .astype(np.float32)
#                 .squeeze()
#             )

#             t0048_bnn_preds_rs = []
#             for i in t0048_bnn_reduced_maps:
#                 t0048_bayes_net.load_map(i)
#                 t0048_bnn_preds_rs.append(
#                     t0048_bayes_net.predict(
#                         np.concatenate([sts, acts], axis=-1), map_only=True
#                     )
#                     .astype(np.float32)
#                     .squeeze()
#                 )

#             t0048_t0012_bnn_preds_rs = []
#             for i in t0048_t0012_bnn_reduced_maps:
#                 t0048_bayes_net.load_map(i)
#                 t0048_t0012_bnn_preds_rs.append(
#                     t0048_bayes_net.predict(
#                         np.concatenate([sts, acts], axis=-1), map_only=True
#                     )
#                     .astype(np.float32)
#                     .squeeze()
#                 )

#             speeds = rng.uniform(0.0, 0.44, n_samps)
#             angles = rng.uniform(-180.0, 180.0, n_samps)
#             sample_acts = np.stack([speeds, angles]).T

#             t0012_mlp_r_full = np.zeros(sts.shape[0])
#             t0012_mlp_r_rs = [np.zeros(sts.shape[0]) for i in t0012_mlp_preds_rs]

#             t0012_bnn_r_full = np.zeros(sts.shape[0])
#             t0012_bnn_r_rs = [np.zeros(sts.shape[0]) for i in t0012_bnn_preds_rs]

#             t0012_t0048_bnn_r_full = np.zeros(sts.shape[0])
#             t0012_t0048_bnn_r_rs = [
#                 np.zeros(sts.shape[0]) for i in t0012_t0048_bnn_preds_rs
#             ]

#             t0048_mlp_r_full = np.zeros(sts.shape[0])
#             t0048_mlp_r_rs = [np.zeros(sts.shape[0]) for i in t0048_mlp_preds_rs]

#             t0048_bnn_r_full = np.zeros(sts.shape[0])
#             t0048_bnn_r_rs = [np.zeros(sts.shape[0]) for i in t0048_bnn_preds_rs]

#             t0048_t0012_bnn_r_full = np.zeros(sts.shape[0])
#             t0048_t0012_bnn_r_rs = [
#                 np.zeros(sts.shape[0]) for i in t0048_t0012_bnn_preds_rs
#             ]

#             for i in range(sts.shape[0]):
#                 sts_samps = np.repeat(
#                     sts[i, :].reshape(-1, sts.shape[1]), n_samps, axis=0
#                 )
#                 t0012_mlp_psamps_full = t0012_mlp_full(sts_samps, sample_acts).astype(
#                     np.float32
#                 )
#                 t0012_mlp_psamps_rs = [
#                     n(sts_samps, sample_acts).astype(np.float32) for n in t0012_mlp_rs
#                 ]

#                 t0012_mlp_r_full[i] = t0012_mlp_psamps_full.mean()
#                 for k, j in enumerate(t0012_mlp_r_rs):
#                     j[i] = t0012_mlp_psamps_rs[k].mean()

#                 t0012_bayes_net.load_map(t0012_bnn_full_map)
#                 t0012_bnn_psamps_full = (
#                     t0012_bayes_net.predict(
#                         np.concatenate([sts_samps, sample_acts], axis=-1), map_only=True
#                     )
#                     .astype(np.float32)
#                     .squeeze()
#                 )

#                 t0012_bnn_psamps_rs = []
#                 for n in t0012_bnn_reduced_maps:
#                     t0012_bayes_net.load_map(n)
#                     t0012_bnn_psamps_rs.append(
#                         t0012_bayes_net.predict(
#                             np.concatenate([sts_samps, sample_acts], axis=-1),
#                             map_only=True,
#                         )
#                         .astype(np.float32)
#                         .squeeze()
#                     )

#                 t0012_t0048_bnn_psamps_rs = []
#                 for n in t0012_t0048_bnn_reduced_maps:
#                     t0012_bayes_net.load_map(n)
#                     t0012_t0048_bnn_psamps_rs.append(
#                         t0012_bayes_net.predict(
#                             np.concatenate([sts_samps, sample_acts], axis=-1),
#                             map_only=True,
#                         )
#                         .astype(np.float32)
#                         .squeeze()
#                     )

#                 t0012_bnn_r_full[i] = t0012_bnn_psamps_full.mean()
#                 for k, j in enumerate(t0012_bnn_r_rs):
#                     j[i] = t0012_bnn_psamps_rs[k].mean()

#                 for k, j in enumerate(t0012_t0048_bnn_r_rs):
#                     j[i] = t0012_t0048_bnn_psamps_rs[k].mean()

#                 t0048_mlp_psamps_full = t0048_mlp_full(sts_samps, sample_acts).astype(
#                     np.float32
#                 )
#                 t0048_mlp_psamps_rs = [
#                     n(sts_samps, sample_acts).astype(np.float32) for n in t0048_mlp_rs
#                 ]

#                 t0048_mlp_r_full[i] = t0048_mlp_psamps_full.mean()
#                 for k, j in enumerate(t0048_mlp_r_rs):
#                     j[i] = t0048_mlp_psamps_rs[k].mean()

#                 t0048_bayes_net.load_map(t0048_bnn_full_map)
#                 t0048_bnn_psamps_full = (
#                     t0048_bayes_net.predict(
#                         np.concatenate([sts_samps, sample_acts], axis=-1), map_only=True
#                     )
#                     .astype(np.float32)
#                     .squeeze()
#                 )

#                 t0048_bnn_psamps_rs = []
#                 for n in t0048_bnn_reduced_maps:
#                     t0048_bayes_net.load_map(n)
#                     t0048_bnn_psamps_rs.append(
#                         t0048_bayes_net.predict(
#                             np.concatenate([sts_samps, sample_acts], axis=-1),
#                             map_only=True,
#                         )
#                         .astype(np.float32)
#                         .squeeze()
#                     )

#                 t0048_t0012_bnn_psamps_rs = []
#                 for n in t0048_t0012_bnn_reduced_maps:
#                     t0048_bayes_net.load_map(n)
#                     t0048_t0012_bnn_psamps_rs.append(
#                         t0048_bayes_net.predict(
#                             np.concatenate([sts_samps, sample_acts], axis=-1),
#                             map_only=True,
#                         )
#                         .astype(np.float32)
#                         .squeeze()
#                     )

#                 t0048_bnn_r_full[i] = t0048_bnn_psamps_full.mean()
#                 for k, j in enumerate(t0048_bnn_r_rs):
#                     j[i] = t0048_bnn_psamps_rs[k].mean()

#                 for k, j in enumerate(t0048_t0012_bnn_r_rs):
#                     j[i] = t0048_t0012_bnn_psamps_rs[k].mean()

#             t0012_mlp_c_rewards_full = np.zeros(sts.shape[0] - 1)
#             t0012_mlp_c_rewards_rs = [
#                 np.zeros(sts.shape[0] - 1) for i in t0012_mlp_preds_rs
#             ]

#             t0012_bnn_c_rewards_full = np.zeros(sts.shape[0] - 1)
#             t0012_bnn_c_rewards_rs = [
#                 np.zeros(sts.shape[0] - 1) for i in t0012_bnn_preds_rs
#             ]

#             t0012_t0048_bnn_c_rewards_rs = [
#                 np.zeros(sts.shape[0] - 1) for i in t0012_t0048_bnn_preds_rs
#             ]

#             t0048_mlp_c_rewards_full = np.zeros(sts.shape[0] - 1)
#             t0048_mlp_c_rewards_rs = [
#                 np.zeros(sts.shape[0] - 1) for i in t0048_mlp_preds_rs
#             ]

#             t0048_bnn_c_rewards_full = np.zeros(sts.shape[0] - 1)
#             t0048_bnn_c_rewards_rs = [
#                 np.zeros(sts.shape[0] - 1) for i in t0048_bnn_preds_rs
#             ]

#             t0048_t0012_bnn_c_rewards_rs = [
#                 np.zeros(sts.shape[0] - 1) for i in t0048_t0012_bnn_preds_rs
#             ]

#             for i in range(sts.shape[0] - 1):
#                 t0012_mlp_c_rewards_full[i] = (
#                     t0012_mlp_preds_full[i]
#                     + discount * t0012_mlp_r_full[i + 1]
#                     - t0012_mlp_r_full[i]
#                 )

#                 for k, j in enumerate(t0012_mlp_c_rewards_rs):
#                     j[i] = (
#                         t0012_mlp_preds_rs[k][i]
#                         + discount * t0012_mlp_r_rs[k][i + 1]
#                         - t0012_mlp_r_rs[k][i]
#                     )

#                 t0012_bnn_c_rewards_full[i] = (
#                     t0012_bnn_preds_full[i]
#                     + discount * t0012_bnn_r_full[i + 1]
#                     - t0012_bnn_r_full[i]
#                 )

#                 for k, j in enumerate(t0012_bnn_c_rewards_rs):
#                     j[i] = (
#                         t0012_bnn_preds_rs[k][i]
#                         + discount * t0012_bnn_r_rs[k][i + 1]
#                         - t0012_bnn_r_rs[k][i]
#                     )

#                 for k, j in enumerate(t0012_t0048_bnn_c_rewards_rs):
#                     j[i] = (
#                         t0012_t0048_bnn_preds_rs[k][i]
#                         + discount * t0012_t0048_bnn_r_rs[k][i + 1]
#                         - t0012_t0048_bnn_r_rs[k][i]
#                     )

#                 t0048_mlp_c_rewards_full[i] = (
#                     t0048_mlp_preds_full[i]
#                     + discount * t0048_mlp_r_full[i + 1]
#                     - t0048_mlp_r_full[i]
#                 )

#                 for k, j in enumerate(t0048_mlp_c_rewards_rs):
#                     j[i] = (
#                         t0048_mlp_preds_rs[k][i]
#                         + discount * t0048_mlp_r_rs[k][i + 1]
#                         - t0048_mlp_r_rs[k][i]
#                     )

#                 t0048_bnn_c_rewards_full[i] = (
#                     t0048_bnn_preds_full[i]
#                     + discount * t0048_bnn_r_full[i + 1]
#                     - t0048_bnn_r_full[i]
#                 )

#                 for k, j in enumerate(t0048_bnn_c_rewards_rs):
#                     j[i] = (
#                         t0048_bnn_preds_rs[k][i]
#                         + discount * t0048_bnn_r_rs[k][i + 1]
#                         - t0048_bnn_r_rs[k][i]
#                     )

#                 for k, j in enumerate(t0048_t0012_bnn_c_rewards_rs):
#                     j[i] = (
#                         t0048_t0012_bnn_preds_rs[k][i]
#                         + discount * t0048_t0012_bnn_r_rs[k][i + 1]
#                         - t0048_t0012_bnn_r_rs[k][i]
#                     )
#             t0012_mlp_can_full.append(t0012_mlp_c_rewards_full)
#             t0012_mlp_can_rs.append(t0012_mlp_c_rewards_rs)

#             t0012_bnn_can_full.append(t0012_bnn_c_rewards_full)
#             t0012_bnn_can_rs.append(t0012_bnn_c_rewards_rs)

#             t0012_t0048_bnn_can_rs.append(t0012_t0048_bnn_c_rewards_rs)

#             t0048_mlp_can_full.append(t0048_mlp_c_rewards_full)
#             t0048_mlp_can_rs.append(t0048_mlp_c_rewards_rs)

#             t0048_bnn_can_full.append(t0048_bnn_c_rewards_full)
#             t0048_bnn_can_rs.append(t0048_bnn_c_rewards_rs)

#             t0048_t0012_bnn_can_rs.append(t0048_t0012_bnn_c_rewards_rs)

# t0012_mlp_can_full = np.concatenate(t0012_mlp_can_full)
# t0012_mlp_can_rs = [np.concatenate(x) for x in zip(*t0012_mlp_can_rs)]

# t0012_bnn_can_full = np.concatenate(t0012_bnn_can_full)
# t0012_bnn_can_rs = [np.concatenate(x) for x in zip(*t0012_bnn_can_rs)]

# t0012_t0048_bnn_can_rs = [np.concatenate(x) for x in zip(*t0012_t0048_bnn_can_rs)]

# t0048_mlp_can_full = np.concatenate(t0048_mlp_can_full)
# t0048_mlp_can_rs = [np.concatenate(x) for x in zip(*t0048_mlp_can_rs)]

# t0048_bnn_can_full = np.concatenate(t0048_bnn_can_full)
# t0048_bnn_can_rs = [np.concatenate(x) for x in zip(*t0048_bnn_can_rs)]

# t0048_t0012_bnn_can_rs = [np.concatenate(x) for x in zip(*t0048_t0012_bnn_can_rs)]


# t0012_mlp_epic = np.zeros(len(t0012_mlp_rs))
# t0012_bnn_epic = np.zeros(len(t0012_bnn_reduced_maps))
# t0012_t0048_bnn_epic = np.zeros(len(t0012_t0048_bnn_reduced_maps))

# t0048_mlp_epic = np.zeros(len(t0012_mlp_rs))
# t0048_bnn_epic = np.zeros(len(t0048_bnn_reduced_maps))
# t0048_t0012_bnn_epic = np.zeros(len(t0048_t0012_bnn_reduced_maps))
# epic_full = np.zeros((4, 4))

# for i in range(len(t0012_mlp_rs)):
#     t0012_mlp_epic[i] = np.sqrt(
#         1 - np.corrcoef(t0012_mlp_can_full, t0012_mlp_can_rs[i])[0, 1]
#     ) / np.sqrt(2)

# for i in range(len(t0012_bnn_reduced_maps)):
#     t0012_bnn_epic[i] = np.sqrt(
#         1 - np.corrcoef(t0012_bnn_can_full, t0012_bnn_can_rs[i])[0, 1]
#     ) / np.sqrt(2)

# for i in range(len(t0012_t0048_bnn_reduced_maps)):
#     t0012_t0048_bnn_epic[i] = np.sqrt(
#         1 - np.corrcoef(t0012_bnn_can_full, t0012_t0048_bnn_can_rs[i])[0, 1]
#     ) / np.sqrt(2)

# for i in range(len(t0048_mlp_rs)):
#     t0048_mlp_epic[i] = np.sqrt(
#         1 - np.corrcoef(t0048_mlp_can_full, t0048_mlp_can_rs[i])[0, 1]
#     ) / np.sqrt(2)

# for i in range(len(t0048_bnn_reduced_maps)):
#     t0048_bnn_epic[i] = np.sqrt(
#         1 - np.corrcoef(t0048_bnn_can_full, t0048_bnn_can_rs[i])[0, 1]
#     ) / np.sqrt(2)

# for i in range(len(t0048_t0012_bnn_reduced_maps)):
#     t0048_t0012_bnn_epic[i] = np.sqrt(
#         1 - np.corrcoef(t0048_bnn_can_full, t0048_t0012_bnn_can_rs[i])[0, 1]
#     ) / np.sqrt(2)

# for i, part_a in enumerate(
#     [
#         t0012_mlp_can_full,
#         t0012_bnn_can_full,
#         t0048_mlp_can_full,
#         t0048_bnn_can_full,
#     ]
# ):
#     for j, part_b in enumerate(
#         [
#             t0012_mlp_can_full,
#             t0012_bnn_can_full,
#             t0048_mlp_can_full,
#             t0048_bnn_can_full,
#         ]
#     ):
#         if i == j:
#             epic_full[i, j] = 0.0
#         else:
#             epic_full[i, j] = np.sqrt(1 - np.corrcoef(part_a, part_b)[0, 1]) / np.sqrt(
#                 2
#             )

# np.save("t0012_mlp_epic", t0012_mlp_epic)


# np.save("t0012_bnn_epic", t0012_bnn_epic)


# np.save("t0012_t0048_bnn_epic", t0012_t0048_bnn_epic)


# np.save("t0048_mlp_epic", t0048_mlp_epic)


# np.save("t0048_bnn_epic", t0048_bnn_epic)


# np.save("t0048_t0012_bnn_epic", t0048_t0012_bnn_epic)


# np.save("epic_full", epic_full)

In [ ]:
t0012_mlp_epic = np.load("t0012_mlp_epic.npy")
t0012_bnn_epic = np.load("t0012_bnn_epic.npy")
t0012_t0048_bnn_epic = np.load("t0012_t0048_bnn_epic.npy")

t0048_mlp_epic = np.load("t0048_mlp_epic.npy")
t0048_bnn_epic = np.load("t0048_bnn_epic.npy")
t0048_t0012_bnn_epic = np.load("t0048_t0012_bnn_epic.npy")

epic_full = np.load("epic_full.npy")

In [ ]:
epic_mlp_mean = [t0012_mlp_epic.mean(), t0048_mlp_epic.mean()]
epic_mlp_std = [t0012_mlp_epic.std(ddof=1), t0048_mlp_epic.std(ddof=1)]

epic_bnn_mean = [t0012_bnn_epic.mean(), t0048_bnn_epic.mean()]
epic_bnn_std = [t0012_bnn_epic.std(ddof=1), t0048_bnn_epic.std(ddof=1)]

epic_bnn_p_prior_mean = [t0012_t0048_bnn_epic.mean(), t0048_t0012_bnn_epic.mean()]
epic_bnn_p_prior_std = [
    t0012_t0048_bnn_epic.std(ddof=1),
    t0048_t0012_bnn_epic.std(ddof=1),
]

In [ ]:
participants = ["t0012", "t0048"]
epic_mlp = {
    "Participant": participants,
    "Mean EPIC Distance": epic_mlp_mean,
    "Std EPIC Distance": epic_mlp_std,  # Precomputed error for each category
}
epic_mlp_df = pd.DataFrame(epic_mlp)
epic_mlp_df["Reward Function"] = "MLP"

epic_bnn = {
    "Participant": participants,
    "Mean EPIC Distance": epic_bnn_mean,
    "Std EPIC Distance": epic_bnn_std,  # Precomputed error for each category
}
epic_bnn_df = pd.DataFrame(epic_bnn)
epic_bnn_df["Reward Function"] = "BNN"

epic_bnn_p_prior = {
    "Participant": participants,
    "Mean EPIC Distance": epic_bnn_p_prior_mean,
    "Std EPIC Distance": epic_bnn_p_prior_std,  # Precomputed error for each category
}
epic_bnn_p_prior_df = pd.DataFrame(epic_bnn_p_prior)
epic_bnn_p_prior_df["Reward Function"] = "BNN (P-Prior)"

df = pd.concat([epic_mlp_df, epic_bnn_df, epic_bnn_p_prior_df])

# Create the Seaborn barplot without default error bars
r_ax = sns.barplot(
    x="Participant",
    y="Mean EPIC Distance",
    hue="Reward Function",
    data=df,
    errorbar=None,
)

# Add custom error bars using Matplotlib
# Get the x-positions of the bars for errorbar placement
n_rfs = len(df["Reward Function"].unique())
bar_width = 0.8 / n_rfs
offsets = np.linspace(-bar_width * (n_rfs - 1) / 2, bar_width * (n_rfs - 1) / 2, n_rfs)
datasets = r_ax.get_xticks()
rf_order = df["Reward Function"].unique()

for i, dataset_label in enumerate(df["Participant"].unique()):
    for j, rf_label in enumerate(rf_order):
        subset = df[
            (df["Participant"] == dataset_label) & (df["Reward Function"] == rf_label)
        ]
        if not subset.empty:
            x_pos = datasets[i] + offsets[j]
            y_val = subset["Mean EPIC Distance"].iloc[0]
            y_err = subset["Std EPIC Distance"].iloc[0]
            r_ax.errorbar(
                x_pos, y_val, yerr=y_err, fmt="none", color="black", capsize=5
            )

plt.xticks(rotation=30)
plt.title("EPIC Distance Means: Full Data Models vs 99.9% Reduced Data Models")
plt.ylabel("Mean EPIC Distance")
plt.show()

In [ ]:
participants = ["t0012", "t0048"]
epic_mlp_df = pd.DataFrame(np.array([t0012_mlp_epic, t0048_mlp_epic]).T)
epic_mlp_df.columns = participants
epic_mlp_df = epic_mlp_df.melt(var_name="Participant", value_name="EPIC Distance")
epic_mlp_df["Reward Function"] = "MLP"

epic_bnn_df = pd.DataFrame(np.array([t0012_bnn_epic, t0048_bnn_epic]).T)
epic_bnn_df.columns = participants
epic_bnn_df = epic_bnn_df.melt(var_name="Participant", value_name="EPIC Distance")
epic_bnn_df["Reward Function"] = "BNN"

epic_bnn_p_prior_df = pd.DataFrame(
    np.array([t0012_t0048_bnn_epic, t0048_t0012_bnn_epic]).T
)
epic_bnn_p_prior_df.columns = participants
epic_bnn_p_prior_df = epic_bnn_p_prior_df.melt(
    var_name="Participant", value_name="EPIC Distance"
)
epic_bnn_p_prior_df["Reward Function"] = "BNN (P-Prior)"

df = pd.concat([epic_mlp_df, epic_bnn_df, epic_bnn_p_prior_df])

sns.violinplot(
    x="Participant",
    y="EPIC Distance",
    hue="Reward Function",
    data=df,
    cut=0,
)
plt.title("EPIC Distance Distributions: Full Data Models vs 99.9% Reduced Data Models")
plt.ylabel("EPIC Distance")
plt.show()

In [ ]:
mask = np.triu(np.ones_like(epic_full))
hlabels = ["t0012: MLP", "t0012: BNN", "t0048: MLP", "t0048: BNN"]
sns.heatmap(
    epic_full,
    cmap="viridis_r",
    xticklabels=hlabels,
    yticklabels=hlabels,
    annot=True,
    fmt=".3g",
    mask=mask,
)
plt.title("EPIC Distances: Comparing Full Data Models")
plt.show()